# Cipher — Hosted on Colab Pro+ A100

Spins up Cipher (kin-cipher) on this Colab GPU and exposes it via cloudflared tunnel.
Use the printed URL from your laptop for fast generation.

**Runtime needed:** A100 (Colab Pro+). At T4 it'll work but slower (~5-10 t/s).

In [ ]:
# Cell 1 — install Ollama + cloudflared + curl
import os, subprocess, time

print('Installing Ollama...')
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

print('Downloading cloudflared...')
subprocess.run('wget -qq https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)

print('Done.')

In [ ]:
# Cell 2 — start Ollama serve in background
import subprocess, time, os

# kill any existing
subprocess.run('pkill -f ollama 2>/dev/null', shell=True)
time.sleep(2)

# start Ollama with GPU support
subprocess.Popen('OLLAMA_FLASH_ATTENTION=1 OLLAMA_KEEP_ALIVE=24h nohup ollama serve > /tmp/ollama.log 2>&1 &', shell=True)
time.sleep(8)

# verify
import urllib.request
print(urllib.request.urlopen('http://localhost:11434/api/version').read().decode())

In [ ]:
# Cell 3 — pull Cipher GGUF (public — no auth needed)
import subprocess
print('Pulling Cipher GGUF (~18GB, 5-10 min on Colab fast network)...')
result = subprocess.run('ollama pull hf.co/Auroraventures/cipher-sft-merged-Q4_K_M-GGUF', shell=True, capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

# Tag as kin-cipher
subprocess.run('ollama cp hf.co/Auroraventures/cipher-sft-merged-Q4_K_M-GGUF kin-cipher', shell=True)
print('---')
subprocess.run('ollama list', shell=True)

In [ ]:
# Cell 4 — quick test (no tunnel yet, just verify model works on this GPU)
import urllib.request, json, time

prompt = '<start_of_turn>user\nIn 30 words, describe yourself.<end_of_turn>\n<start_of_turn>model\n'
body = json.dumps({
    'model': 'kin-cipher',
    'prompt': prompt,
    'raw': True,
    'stream': False,
    'options': {'temperature': 0.7, 'num_predict': 100, 'stop': ['<end_of_turn>']},
}).encode()

t0 = time.time()
req = urllib.request.Request('http://localhost:11434/api/generate', data=body, headers={'Content-Type': 'application/json'})
data = json.loads(urllib.request.urlopen(req, timeout=300).read())
elapsed = time.time() - t0
print(f'Time: {elapsed:.1f}s | tokens: {data.get("eval_count")} | speed: {data.get("eval_count",0)/(data.get("eval_duration",1)/1e9):.1f} t/s')
print('---')
print(data['response'])

In [ ]:
# Cell 5 — start cloudflared tunnel and print public URL
import subprocess, time, re, os

subprocess.run('pkill -f cloudflared 2>/dev/null', shell=True)
time.sleep(2)

subprocess.Popen('nohup cloudflared tunnel --url http://localhost:11434 > /tmp/tunnel.log 2>&1 &', shell=True)

# Wait for tunnel URL to appear in log
url = None
for _ in range(30):
    time.sleep(2)
    try:
        log = open('/tmp/tunnel.log').read()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
        if m:
            url = m.group(0)
            break
    except: pass

if url:
    print(f'\n========================================')
    print(f'CIPHER PUBLIC URL: {url}')
    print(f'========================================')
    print(f'\nTest it:')
    print(f'curl {url}/api/version')
    print(f'\nFrom your laptop, set CIPHER_API_URL to use it.')
else:
    print('Tunnel URL not found - check /tmp/tunnel.log')
    print(open('/tmp/tunnel.log').read())